# C.3 多試幾次能找到正確答案嗎？


只生成一次可能答錯，多生成幾份候選是否有幫助？假設每次答對的機率是0.3，而且各次成功互相獨立。一次錯的機率是0.7，兩次都錯是0.7×0.7=0.49，所以兩次中至少一次對的機率是1−0.49=0.51。前置是[機率與乘法](https://birdhackor.github.io/tiny-perceptron-vlm/first-steps.html#W.4)與[生成採樣](https://birdhackor.github.io/tiny-perceptron-vlm/1.15.html)；這裡先算候選包含正解的機會，還沒決定最後交哪一份。

一般寫成 `1-(1-p)**n`：p是單次成功率，n是候選數，`**`代表次方。先算n次全部失敗，再從1扣掉，就是至少一次成功。這種候選集合包含正解的比例可稱coverage；語言模型評測也常用pass@k描述k個候選裡至少一個通過，但要依指定評測方法計算與標明採樣設定。它不是最後選出的答案正確率。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
p = 0.3
for n in (1, 2, 4, 8):
    coverage = 1 - (1 - p) ** n
    print("候選數", n, "獨立假設下包含正解", round(coverage, 6))
candidates = [3, 4, 3, 5]
print("本組候選", candidates, "包含真值4", 4 in candidates)

前四行應是0.3、0.51、0.7599、0.942352；n增加時，至少出現一次正解的機率上升。最後一行是True，因為固定候選第二項是4。`in`只檢查清單是否含4，沒有選出答案，也沒有生成候選。本程式的0.3是我們假設的數字，清單也是人寫的，不能冒充模型實驗或測得的pass@k。

獨立假設很關鍵。若每次都使用相同的確定性生成，固定問題反覆得到同一個錯答案，就沒有公式期待的新機會。即使使用隨機採樣，各次也可能共享相同誤解，或只換措辭而保持同一錯誤。實際需要保存多題的候選，逐份依真值檢查，再統計集合命中比例；只看一組重複答案，無法單獨估計所有樣本之間的相關性。

多生成也要花更多token與時間。token是模型一次處理或生成的文字片段，可能是一個字、字的一部分或數個字；這裡只需知道更多候選通常需要生成更多這種片段。找到了正解也不代表選得中。這組 `[3,4,3,5]`若最後選第一份，就仍會答3；對每組都知道真值才從中挑對，則是假設有完美選擇能力，不一定能用在未知答案的任務。下一節會讓多數決與算術驗證器選同一組候選，看到集合命中和交出答案的差別。

練習只把candidates改成 `[3,3,3,3]`，p和前面公式不變。先預測最後一行變False，但公式四行完全相同，再執行核對。請說明：固定清單改了，並沒有重新量出單次p；公式的假設也不會自動成為這份候選清單的實測事實。
